# Selecting with loc and iloc

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/y-rosenthal/yr-pre-ml/blob/main/jb/chapters/09-loc-and-iloc.ipynb)

So far, we selected whole columns with `df["col"]`. But a manager often asks about one store, one value, or a
block of rows and columns: "How many employees does store S3 have?" pandas has two tools for this. `loc` selects
by **label** (the name of a row), and `iloc` selects by **position** (where the row is). In this chapter, we learn
both, and when they give different answers.

**In this chapter, you will learn to:**

- explain the difference between a row's label and its position;
- select a row, a single value, or a block of rows and columns by label, with `loc`;
- select the same things by position, with `iloc`;
- remember which slices include their end and which do not;
- see why `loc` and `iloc` give different rows after the rows are sorted;
- change one value of a table with `loc`.

In [ ]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/y-rosenthal/yr-pre-ml/main/data/"

## Labels and positions

We start with a small table of five stores. This time, we give the rows names: `index=[...]` sets the **index**, the
labels of the rows, as it did for a Series in the chapter *Series and DataFrames*.

In [ ]:
stores = pd.DataFrame(
    {
        "city": ["Boston", "Albany", "Miami", "Tampa", "Denver"],
        "region": ["North", "North", "South", "South", "West"],
        "employees": [12, 8, 20, 15, 9],
        "sales": [52000, 31000, 87000, 64000, 40000],
    },
    index=["S1", "S2", "S3", "S4", "S5"],
)
stores

The labels `S1` to `S5` appear on the left, where we saw 0, 1, 2, ... before. The column `sales` holds each
store's monthly sales, in dollars.

Every row now has two ways to name it:

- its **label**: the name in the index, for example `"S3"`;
- its **position**: where it is in the table, counting from 0, as in a list. `S3` is at position 2.

The columns have both too: the column `employees` has the label `"employees"` and the position 2.

## Making a column the index

Data files usually have the store names in an ordinary column. `set_index("store")` makes that column the index:

In [ ]:
table = pd.DataFrame({"store": ["S1", "S2", "S3"], "employees": [12, 8, 20]})
table = table.set_index("store")
table

The column `store` became the index, so the table has one column left, `employees`. The word `store` above the
labels is the name of the index. `set_index` does not change `table` itself: it returns a new DataFrame, so we
assign the result back to `table` to keep it.

## One row by label: `loc`

`stores.loc["S3"]` selects the row with the label `"S3"`:

In [ ]:
stores.loc["S3"]

We get one row, as a Series. Its index holds the column names, and its name (`Name: S3`) is the row's label. Its
dtype is `object` because the row mixes text (`Miami`) and numbers.

Note the square brackets: `loc` is not a function, so we write `stores.loc[...]`, not `stores.loc(...)`.

> **Common mistake:** writing `stores["S3"]` to get a row. Square brackets alone select **columns**, so pandas
> looks for a column named `S3` and stops with `KeyError: 'S3'`. To select rows by label, use `loc`.

## One value: row, then column

To get one value, give `loc` the row label and the column label, separated by a comma. The row comes first, as in
`a[row, col]` for the two-dimensional arrays of the chapter *Two-dimensional arrays*:

In [ ]:
stores.loc["S3", "employees"]

Store S3 has 20 employees.

> **Note:** Colab shows the value as `np.int64(20)`: the number 20, stored as a NumPy whole number, as in the
> chapter *Indexing, slicing and arithmetic*. `print()` shows just `20`.

## Several rows and columns

To select several rows or several columns, give `loc` a **list** of labels in each place:

In [ ]:
stores.loc[["S1", "S4"], ["city", "sales"]]

The result is a smaller DataFrame: the rows `S1` and `S4`, and the columns `city` and `sales`, in the order of the
lists. As with `df[["a", "b"]]`, the inner brackets make a list.

To keep every row, write a colon `:` in the row place. `stores.loc[:, ["city", "sales"]]` gives all five stores,
with two columns.

> **Try it:** show the region and the employees of the stores S2, S3 and S5.

## A label slice includes the end

We can also slice by label, with a colon between two labels:

In [ ]:
stores.loc["S2":"S4"]

We get **three** rows: `S2`, `S3` and `S4`. A label slice **includes** its end. This is different from a slice of
a Python list, which stops just before the end. It is how we talk about names: "the stores from S2 to S4" includes
S4. And with labels, there is no simple way to say "the label after S4", so pandas lets us name the last row we
want.

## Positions: `iloc`

`iloc` (the "i" stands for *integer*) selects by position, counting from 0. `stores.iloc[0]` is the first row,
whatever its label:

In [ ]:
stores.iloc[0]

It is the row of store `S1` (see `Name: S1`). For one value, give the row position and then the column position:

In [ ]:
stores.iloc[0, 2]

Row 0 is `S1`, and column 2 is `employees` (the columns are counted from 0 too: `city` is 0, `region` is 1), so
we get 12, the employees of store S1.

## A position slice excludes the end

A slice of positions works exactly like a slice of a list or an array: it **stops before** the end.

In [ ]:
stores.iloc[1:3]

Positions 1 and 2: the stores `S2` and `S3`. Position 3 is not included.

The same works for columns. A colon alone in the row place keeps every row:

In [ ]:
stores.iloc[:, 0:2]

Every row, and the columns at positions 0 and 1: `city` and `region`.

Here are `loc` and `iloc` side by side, on the `stores` table:

| What we want | By label: `loc` | By position: `iloc` |
|---|---|---|
| one row | `stores.loc["S3"]` | `stores.iloc[2]` |
| one value | `stores.loc["S3", "employees"]` | `stores.iloc[2, 2]` |
| rows S2 to S4 | `stores.loc["S2":"S4"]` (includes `S4`) | `stores.iloc[1:4]` (stops before 4) |
| every row, two columns | `stores.loc[:, ["city", "region"]]` | `stores.iloc[:, 0:2]` |

> **Common mistake:** mixing them up. `stores.iloc["S3"]` fails, because `iloc` takes only whole numbers, and
> `stores.loc[2]` fails with `KeyError: 2`, because no row has the label `2`.

## `loc` and `iloc` on a data file

Now let's read a real data file, `sales.csv`: 2,000 sales of a store chain, one row for each sale (the chapter
*Reading and describing data* introduced it).

In [ ]:
sales = pd.read_csv(DATA_URL + "sales.csv")
sales.head()

`read_csv` gives the rows the labels 0, 1, 2, ..., so each row's label is the same as its position. Here,
`sales.loc[0]` and `sales.iloc[0]` give the same row: the first sale of the year, 3 Clothing units in store S1 at
$46.99 each. So in a fresh table, `loc` and `iloc` look alike.

## After sorting, labels and positions differ

Labels stay with their rows. When we sort or filter the rows, the labels move with them, and then a label is no
longer the same as a position. Let's sort the sales by price, from the cheapest. `sort_values("price")` returns the
rows in order of their price (the chapter *New columns and sorting* explains sorting):

In [ ]:
by_price = sales.sort_values("price")
by_price.head()

The lowest price is $1.99 a unit. Look at the index: the labels are now 1911, 1626, 125, ... Each row kept the label
it had in `sales`. Now `iloc[0]` and `loc[0]` are different rows:

In [ ]:
by_price.iloc[0]

`iloc[0]` is the row at position 0 of `by_price`: the first row after sorting, a sale at $1.99 a unit, with the label 1911.

In [ ]:
by_price.loc[0]

`loc[0]` is the row with the **label** 0: the first sale of the year, at $46.99. It is somewhere in the middle of
`by_price`, but `loc` finds it by its name.

So, a simple rule:

- use `loc` when you know **which** row you want (its label);
- use `iloc` when you know **where** the row is (its position), for example "the first five rows".

## Changing one value with `loc`

`loc` can also change a value. Store S3 hired five more people, so it now has 25 employees. We put the `loc`
selection on the left of `=`:

In [ ]:
stores.loc["S3", "employees"] = 25
stores

The value in row `S3`, column `employees`, is now 25. The other values did not change.

Put the row and the column in **one** pair of brackets, as here. The chapter *Filtering rows* uses the same form to
change many rows at once, and explains why other forms can fail.

> **Try it:** Denver (store S5) moved to a new mall, and its monthly sales are now 46000. Change the value, and
> show `stores` again.

## Summary

- Every row has a **label** (its name in the index) and a **position** (0, 1, 2, ...).
  `index=[...]` or `set_index("col")` sets the labels.
- `df.loc[...]` selects by label: `df.loc["S3"]` (one row), `df.loc["S3", "employees"]` (one value),
  `df.loc[["S1", "S4"], ["city", "sales"]]` (a block), `df.loc[:, ["city", "sales"]]` (every row).
- `df.iloc[...]` selects by position: `df.iloc[0]`, `df.iloc[0, 2]`, `df.iloc[1:3]`, `df.iloc[:, 0:2]`.
- A **label slice includes** its end (`df.loc["S2":"S4"]`); a **position slice excludes** it (`df.iloc[1:3]`).
- After sorting or filtering, the labels stay with their rows, so `loc` and `iloc` give different rows.
- `df.loc[row, "col"] = value` changes one value.

## Practice questions

### Question 1 (Easy)

Which store is at position 3 of `stores`? Do `stores.iloc[3]` and `stores.loc["S3"]` give the same row?

<details class="yr-answer">
<summary>Answer</summary>

Position 3 is the fourth row, store `S4` (positions count from 0). So `stores.iloc[3]` is the row of `S4`, and
`stores.loc["S3"]` is the row of `S3`: two different rows.

```python
print(stores.iloc[3])
print(stores.loc["S3"])
```

</details>

In [ ]:
# Write your answer here

### Question 2 (Easy)

Without running the code, say how many rows each line gives. Then check.

```python
stores.loc["S1":"S3"]
stores.iloc[1:3]
```

<details class="yr-answer">
<summary>Answer</summary>

The label slice includes its end, so `stores.loc["S1":"S3"]` gives 3 rows (`S1`, `S2`, `S3`). The position slice
stops before its end, so `stores.iloc[1:3]` gives 2 rows (positions 1 and 2: `S2` and `S3`).

```python
print(len(stores.loc["S1":"S3"]))
print(len(stores.iloc[1:3]))
```

</details>

In [ ]:
# Write your answer here

### Question 3 (Medium)

Show the city and the monthly sales of the stores S2 and S5, with `loc`. Then get the same table with `iloc`.

<details class="yr-answer">
<summary>Answer</summary>

With `loc`, use lists of labels. With `iloc`, use lists of positions: `S2` and `S5` are at positions 1 and 4, and
the columns `city` and `sales` are at positions 0 and 3.

```python
print(stores.loc[["S2", "S5"], ["city", "sales"]])
print(stores.iloc[[1, 4], [0, 3]])
```

</details>

In [ ]:
# Write your answer here

### Question 4 (Medium)

Show the date, the store and the units of the first three sales in `sales`, once with `loc` and once with `iloc`.
Be careful with the end of each slice.

<details class="yr-answer">
<summary>Answer</summary>

The first three rows have the labels 0, 1 and 2. A label slice includes its end, so `loc` needs `0:2`; a position
slice does not, so `iloc` needs `0:3`. The columns `date`, `store` and `units` are at positions 0, 1 and 4.

```python
print(sales.loc[0:2, ["date", "store", "units"]])
print(sales.iloc[0:3, [0, 1, 4]])
```

</details>

In [ ]:
# Write your answer here

### Question 5 (Medium)

Show the last two rows and the last two columns of `stores` with `iloc`. (Hint: negative positions work as they do
for lists and arrays.)

<details class="yr-answer">
<summary>Answer</summary>

`-2:` means "from the second to last to the end", for the rows and for the columns.

```python
stores.iloc[-2:, -2:]
```

</details>

In [ ]:
# Write your answer here

### Question 6 (Hard)

Find the sale with the most units in `sales`: sort the sales by units with `sort_values("units")`, and take the
last row of the sorted table. What is its label? Then use that label with `loc` on `sales` to show the same row.

<details class="yr-answer">
<summary>Answer</summary>

The last row has the position `-1`, so we use `iloc[-1]`. The `Name` of the row it returns is its label, which we
can then give to `loc`. The sale with the most units has the label 1449: 16 Grocery units in store S1, on
September 19.

```python
by_units = sales.sort_values("units")
most_units = by_units.iloc[-1]
print(most_units)
print(sales.loc[most_units.name])
```

</details>

In [ ]:
# Write your answer here

### Question 7 (Hard)

Store S2 hired 3 more people. Add 3 to its number of employees with `loc`, without typing the new number yourself.

<details class="yr-answer">
<summary>Answer</summary>

Read the old value with `loc`, add 3, and put the result back with `loc`. Store S2 had 8 employees, so it now has
11.

```python
stores.loc["S2", "employees"] = stores.loc["S2", "employees"] + 3
stores
```

</details>

In [ ]:
# Write your answer here

### Question 8 (Challenge)

Write a loop over the labels in `stores.index` that prints, for each store, its city and its monthly sales per
employee, rounded to whole dollars. Use `loc` inside the loop.

<details class="yr-answer">
<summary>Answer</summary>

`stores.index` holds the labels, so each `label` can go into `loc`. The sales per employee is the sales divided by
the employees. Denver (store S5) sells the most per employee: $4,444 a month.

```python
for label in stores.index:
    city = stores.loc[label, "city"]
    per_employee = stores.loc[label, "sales"] / stores.loc[label, "employees"]
    print(f"{label} {city}: ${per_employee:,.0f} per employee")
```

</details>

In [ ]:
# Write your answer here